## R script

#### <font color=lightblue> As Y-A-S represents compositional data and exhibits strong collinearity among components, an isometric log-ratio (ILR) transformation was applied prior to analysis

In [ ]:
load(yas.r)

In [ ]:
pacman::p_load(compositions, randomForest, rfUtilities, rfPermute, forcats, rfcv)

In [ ]:
as_wider = subset(yas_df_wider, SoilType == "AS")
ns_wider = subset(yas_df_wider, SoilType == "NS")

In [ ]:
rownames(as_df_wider) = NULL
as_ilr = as_df_wider %>% column_to_rownames(var = "sample") %>%
    dplyr::select(`Growth potential`, `Resource acquisition`, `Stress tolerance`) %>%
    acomp %>% ilr %>% as.data.frame() %>% rename(ilr1 = V1, ilr2 = V2) %>% rownames_to_column(var = "sample") %>%
    left_join(as_wider, by = "sample")

rownames(ns_df_wider) = NULL
ns_ilr = ns_df_wider %>% column_to_rownames(var = "sample") %>%
    dplyr::select(`Growth potential`, `Resource acquisition`, `Stress tolerance`) %>%
    acomp %>% ilr %>% as.data.frame() %>% rename(ilr1 = V1, ilr2 = V2) %>% rownames_to_column(var = "sample") %>%
    left_join(ns_wider, by = "sample")

#### <font color=lightblue> Fig.2a

In [ ]:
ggplot(as_ilr, aes(ilr1, ilr2, color = Continents, fill = Continents)) +
    geom_point(size = 2, alpha = 0.5, pch = 16)  +
    theme_bw() +
    my_theme +
    facet_wrap(~SoilType, scales = "free") +
    scale_color_manual(values = c("#e07167", "#b8941b", "#30aa86", 
    "#2ea2cc", "#8d82b6", "#b56496")) +
    scale_fill_manual(values = c("#e07167", "#b8941b", "#30aa86", 
    "#2ea2cc", "#8d82b6", "#b56496")) +
    xlab("ILR1") +
    ylab("ILR2") +
    theme(strip.text = element_text(size = 30), legend.position = "none")

In [ ]:
ggplot(ns_ilr, aes(ilr1, ilr2, color = Continents, fill = Continents)) +
    geom_point(size = 2, alpha = 0.5, pch = 16)  +
    theme_bw() +
    my_theme +
    facet_wrap(~SoilType, scales = "free") +
    scale_color_manual(values = c("#e07167", "#b8941b", "#30aa86", 
    "#2ea2cc", "#8d82b6", "#b56496")) +
    scale_fill_manual(values = c("#e07167", "#b8941b", "#30aa86", 
    "#2ea2cc", "#8d82b6", "#b56496")) +
    xlab("ILR1") +
    ylab("ILR2") +
    theme(strip.text = element_text(size = 30), legend.position = "none")

#### <font color=lightblue> Fig.2b

In [ ]:
as_rf_ilr1_df = as_ilr %>% select(-ilr2) %>% column_to_rownames(var = "sample")
as_rf_ilr2_df = as_ilr %>% select(-ilr1) %>% column_to_rownames(var = "sample")


ns_rf_ilr1_df = ns_ilr %>% select(-ilr2) %>% column_to_rownames(var = "sample")
ns_rf_ilr2_df = ns_ilr %>% select(-ilr1) %>% column_to_rownames(var = "sample")

In [ ]:
model_as1 = rfPermute(ilr1~., data = as_rf_ilr1_df, ntree = 1000, importance = T, nrep = 1000, num.cores = 20)
model_as2 = rfPermute(ilr2~., data = as_rf_ilr2_df, ntree = 1000, importance = T, nrep = 1000, num.cores = 20)
model_ns1 = rfPermute(ilr1~., data = ns_rf_ilr1_df, ntree = 1000, importance = T, nrep = 1000, num.cores = 20)
model_ns2 = rfPermute(ilr2~., data = ns_rf_ilr2_df, ntree = 1000, importance = T, nrep = 1000, num.cores = 20)

In [ ]:
cv_as1 = rfcv(trainx = as_rf_ilr1_df[,-1], trainy = as_rf_ilr1_df[,1], cv.fold = 10)
cv_as2 = rfcv(trainx = as_rf_ilr2_df[,-1], trainy = as_rf_ilr2_df[,1], cv.fold = 10)
cv_ns1 = rfcv(trainx = ns_rf_ilr1_df[,-1], trainy = ns_rf_ilr1_df[,1], cv.fold = 10)
cv_ns2 = rfcv(trainx = ns_rf_ilr2_df[,-1], trainy = ns_rf_ilr2_df[,1], cv.fold = 10)

In [ ]:
importance(model_as1) %>% as.data.frame %>% arrange(desc(`%IncMSE`)) %>% head(10) %>%
    mutate(sign = ifelse(`%IncMSE.pval`<0.05, "sign", "no sign"),
    label = case_when(`%IncMSE.pval`<0.05 & `%IncMSE.pval`> 0.01 ~ "*",
                        `%IncMSE.pval`<0.01 & `%IncMSE.pval`> 0.001 ~ "**",
                        `%IncMSE.pval`<0.001 ~ "***", TRUE ~ "")) %>%
    rownames_to_column(var = "variable") %>%
    mutate(SoilType = "AS") %>%
    ggplot(aes(reorder(variable, `%IncMSE`), `%IncMSE`)) +
    geom_bar(stat = "identity") +
    scale_y_continuous(expand = c(0,0), limits = c(0, 55)) +
    coord_flip() +
    theme_bw() +
    facet_wrap(~SoilType) +
    xlab("") +
    ylab("%IncMSE on ILR1") +
    my_theme +
    theme(strip.text = element_text(size = 25),
    legend.position = "none",strip.background = element_rect(fill = "gray90"),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    panel.border = element_rect(color = "black", linewidth = 1))

importance(model_as2) %>% as.data.frame %>% arrange(desc(`%IncMSE`)) %>% head(10) %>%
    mutate(sign = ifelse(`%IncMSE.pval`<0.05, "sign", "no sign"),
    label = case_when(`%IncMSE.pval`<0.05 & `%IncMSE.pval`> 0.01 ~ "*",
                        `%IncMSE.pval`<0.01 & `%IncMSE.pval`> 0.001 ~ "**",
                        `%IncMSE.pval`<0.001 ~ "***", TRUE ~ "")) %>%
    rownames_to_column(var = "variable") %>%
    mutate(SoilType = "AS") %>%
    ggplot(aes(reorder(variable, `%IncMSE`), `%IncMSE`)) +
    geom_bar(stat = "identity") +
    scale_y_continuous(expand = c(0,0), limits = c(0, 55)) +
    coord_flip() +
    theme_bw() +
    facet_wrap(~SoilType) +
    xlab("") +
    ylab("%IncMSE on ILR2") +
    my_theme +
    theme(strip.text = element_text(size = 25),
    legend.position = "none",strip.background = element_rect(fill = "gray90"),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    panel.border = element_rect(color = "black", linewidth = 1))


importance(model_ns1) %>% as.data.frame %>% arrange(desc(`%IncMSE`)) %>% head(10) %>%
    mutate(sign = ifelse(`%IncMSE.pval`<0.05, "sign", "no sign"),
    label = case_when(`%IncMSE.pval`<0.05 & `%IncMSE.pval`> 0.01 ~ "*",
                        `%IncMSE.pval`<0.01 & `%IncMSE.pval`> 0.001 ~ "**",
                        `%IncMSE.pval`<0.001 ~ "***", TRUE ~ "")) %>%
    rownames_to_column(var = "variable") %>%
    mutate(SoilType = "NS") %>%
    ggplot(aes(reorder(variable, `%IncMSE`), `%IncMSE`)) +
    geom_bar(stat = "identity") +
    scale_y_continuous(expand = c(0,0), limits = c(0, 55)) +
    coord_flip() +
    theme_bw() +
    facet_wrap(~SoilType) +
    xlab("") +
    ylab("%IncMSE on ILR1") +
    my_theme +
    theme(strip.text = element_text(size = 25),
    legend.position = "none",strip.background = element_rect(fill = "gray90"),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    panel.border = element_rect(color = "black", linewidth = 1))

importance(model_ns2) %>% as.data.frame %>% arrange(desc(`%IncMSE`)) %>% head(10) %>%
    mutate(sign = ifelse(`%IncMSE.pval`<0.05, "sign", "no sign"),
    label = case_when(`%IncMSE.pval`<0.05 & `%IncMSE.pval`> 0.01 ~ "*",
                        `%IncMSE.pval`<0.01 & `%IncMSE.pval`> 0.001 ~ "**",
                        `%IncMSE.pval`<0.001 ~ "***", TRUE ~ "")) %>%
    rownames_to_column(var = "variable") %>%
    mutate(SoilType = "NS") %>%
    ggplot(aes(reorder(variable, `%IncMSE`), `%IncMSE`)) +
    geom_bar(stat = "identity") +
    scale_y_continuous(expand = c(0,0), limits = c(0, 55)) +
    coord_flip() +
    theme_bw() +
    facet_wrap(~SoilType) +
    xlab("") +
    ylab("%IncMSE on ILR2") +
    my_theme +
    theme(strip.text = element_text(size = 25),
    legend.position = "none",strip.background = element_rect(fill = "gray90"),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    panel.border = element_rect(color = "black", linewidth = 1))

#### <font color=lightblue> Fig.2c

#### <font color=lightblue> We developed a custom function to estimate the threshold value

In [ ]:
threshold_analysis <- function(data, x, y, md_cutoff = 12, nboot = 1000, seed = 1122) {

  #preparation
  set.seed(seed)

  df = data[, c(x, y)]
  colnames(df) = c("x", "y")

  #remove outliers (Mahalanobis)
  cov_df = cov(df)
  center = colMeans(df)
  md = mahalanobis(df, center, cov_df)
  df = df[md < md_cutoff, ]

  #candidate smooth models
  lm_min  = lm(y ~ x, data = df)
  lm_quad = lm(y ~ x + I(x^2), data = df)
  gam_mod = mgcv::gam(y ~ s(x, bs = "cs"), data = df)

  AIC_base = data.frame(Model = c("Linear", "Quadratic", "GAM"),
    AIC = c(AIC(lm_min), AIC(lm_quad), AIC(gam_mod)))

  best_base = AIC_base$Model[which.min(AIC_base$AIC)]
    cat("Best base model:", best_base, "\n")

  #threshold models
  step_mod = try(
    chngptm::chngptm(y ~ 1, ~ x, data = df,family = "gaussian", type = "step"), silent = TRUE)

  seg_mod = try(
    segmented::segmented(lm(y ~ x, data = df),seg.Z = ~ x,psi = mean(df$x)),
    silent = TRUE)

  steg_mod = try(
    chngptm::chngptm(y ~ x, ~ x, family = "gaussian", type = "stegmented", data = df),
    silent = TRUE)

  get_aic_safe = function(m) if (inherits(m, "try-error")) NA else AIC(m)

  AIC_thresh = data.frame(
    Model = c("step", "segmented", "stegmented"),
    AIC = c(get_aic_safe(step_mod), get_aic_safe(seg_mod), get_aic_safe(steg_mod)))

  best_thresh = AIC_thresh$Model[which.min(AIC_thresh$AIC)]

  #extract threshold
  get_threshold = function(model, type) {
    if (inherits(model, "try-error")) return(NA)
    if (type %in% c("step", "stegmented")) return(model$chngpt)
    if (type == "segmented") return(model$psi[2])
  }

  thr_est <- switch(
    best_thresh,
    step       = get_threshold(step_mod, "step"),
    segmented  = get_threshold(seg_mod, "segmented"),
    stegmented = get_threshold(steg_mod, "stegmented")
  )

  #bootstrap threshold
  thresholds = numeric(nboot)

  for (i in seq_len(nboot)) {
    samp = df[sample(nrow(df), replace = TRUE), ]

    m = try(
      switch(
        best_thresh,
        segmented =
          segmented::segmented(
            lm(y ~ x, data = samp),
            seg.Z = ~ x,
            psi = mean(samp$x)
          ),
        step =
          chngptm::chngptm(y ~ 1, ~ x,
                           family = "gaussian",
                           type = "step",
                           data = samp),
        stegmented =
          chngptm::chngptm(y ~ x, ~ x,
                           family = "gaussian",
                           type = "stegmented",
                           data = samp)
      ),
      silent = TRUE
    )

    thresholds[i] = if (inherits(m, "try-error")) NA else
      if (best_thresh == "segmented") m$psi[2] else m$chngpt
  }

  thresholds = thresholds[!is.na(thresholds)]

  thr_mean = mean(thresholds)
  thr_ci = quantile(thresholds, c(0.025, 0.975))

  #left / right linear models
  left  = df[df$x <  thr_mean, ]
  right = df[df$x >= thr_mean, ]

  lm_left  = lm(y ~ x, data = left)
  lm_right = lm(y ~ x, data = right)

  #output
  return(list(
    data_used      = df,
    AIC_base       = AIC_base,
    best_base      = best_base,
    AIC_threshold  = AIC_thresh,
    best_threshold = best_thresh,
    threshold_raw  = thr_est,
    threshold_boot = thresholds,
    threshold_mean = thr_mean,
    threshold_ci   = thr_ci,
    lm_left        = lm_left,
    lm_right       = lm_right,
    p_left         = summary(lm_left)$coefficients[2, 4],
    p_right        = summary(lm_right)$coefficients[2, 4]
  ))
}


In [ ]:
as_ilr1_threshold = threshold_analysis(as_rf_ilr1_df, "MAT", "ilr1", md_cutoff = 12, nboot = 1000)
as_ilr2_threshold = threshold_analysis(as_rf_ilr2_df, "MAT", "ilr2", md_cutoff = 12, nboot = 1000)

ns_ilr1_threshold = threshold_analysis(ns_rf_ilr1_df, "MDTR", "ilr1", md_cutoff = 12, nboot = 1000)
ns_ilr2_threshold = threshold_analysis(ns_rf_ilr2_df, "MDTR", "ilr2", md_cutoff = 12, nboot = 1000)

In [ ]:
ggplot() +
    geom_point(data = as_rf_ilr1_df, aes(MAT, ilr1), alpha = 0.7, color = AS, size = 3, pch = 21) +
    geom_vline(xintercept = as_ilr1_threshold$threshold_mean, linetype = "dashed", colour = "red", linewidth = 1.5) +
    geom_smooth(data = as_rf_ilr1_df %>% filter(MAT < as_ilr1_threshold$threshold_mean), aes(MAT, ilr1), 
    method = "gam", formula = y ~ x, se = FALSE, colour = "grey", linewidth = 2.5) +
    geom_smooth(data = as_rf_ilr1_df %>% filter(MAT >= as_ilr1_threshold$threshold_mean), aes(MAT, ilr1), 
    method = "gam", formula = y ~ x, se = FALSE, colour = "grey", linewidth = 2.5) +
    theme_bw() +
    geom_smooth(data = as_rf_ilr1_df, aes(MAT, ilr1), method = "gam", formula = y ~ s(x), 
    se = FALSE, colour = "#cf9c6a", linewidth = 2, alpha = 0.5)  +
    labs(x = "MAT", y = "ILR1") +
    my_theme 

In [ ]:
ggplot() +
    geom_point(data = as_rf_ilr2_df, aes(MAT, ilr2), alpha = 0.7, color = AS, size = 3, pch = 21) +
    geom_vline(xintercept = as_ilr2_threshold$threshold_mean, linetype = "dashed", colour = "red", linewidth = 1.5) +
    geom_smooth(data = as_rf_ilr2_df %>% filter(MAT < as_ilr2_threshold$threshold_mean), aes(MAT, ilr2), 
    method = "gam", formula = y ~ x, se = FALSE, colour = "grey", linewidth = 2.5) +
    geom_smooth(data = as_rf_ilr2_df %>% filter(MAT >= as_ilr2_threshold$threshold_mean), aes(MAT, ilr2), 
    method = "gam", formula = y ~ x, se = FALSE, colour = "grey", linewidth = 2.5) +
    theme_bw() +
    geom_smooth(data = as_rf_ilr2_df, aes(MAT, ilr2), method = "gam", formula = y ~ s(x), 
    se = FALSE, colour = "#cf9c6a", linewidth = 2, alpha = 0.5)  +
    labs(x = "MAT", y = "ILR2") +
    my_theme 


In [ ]:
ggplot() +
    geom_point(data = ns_rf_ilr1_df, aes(MDTR, ilr1), alpha = 0.7, color = NS, size = 3, pch = 21) +
    geom_vline(xintercept = ns_ilr1_threshold$threshold_mean, linetype = "dashed", colour = "red", linewidth = 1.5) +
    geom_smooth(data = ns_rf_ilr1_df %>% filter(MDTR < ns_ilr1_threshold$threshold_mean), aes(MDTR, ilr1), 
    method = "gam", formula = y ~ x, se = FALSE, colour = "grey", linewidth = 2.5) +
    geom_smooth(data = ns_rf_ilr1_df %>% filter(MDTR >= ns_ilr1_threshold$threshold_mean), aes(MDTR, ilr1), 
    method = "gam", formula = y ~ x, se = FALSE, colour = "grey", linewidth = 2.5) +
    geom_smooth(data = ns_rf_ilr1_df, aes(MDTR, ilr1), method = "gam", formula = y ~ s(x), 
    se = FALSE, colour = "#cf9c6a", linewidth = 2, alpha = 0.5)  +
    theme_bw() +
    labs(x = "MDTR", y = "ILR1") +
    my_theme

In [ ]:
ggplot() +
    geom_point(data = ns_rf_ilr2_df, aes(MDTR, ilr2), alpha = 0.7, color = NS, size = 3, pch = 21) +
    geom_vline(xintercept = ns_ilr2_threshold$threshold_mean, linetype = "dashed", colour = "red", linewidth = 1.5) +
    geom_smooth(data = ns_rf_ilr2_df %>% filter(MDTR < ns_ilr2_threshold$threshold_mean), aes(MDTR, ilr2), 
    method = "gam", formula = y ~ x, se = FALSE, colour = "grey", linewidth = 2.5) +
    geom_smooth(data = ns_rf_ilr2_df %>% filter(MDTR >= ns_ilr2_threshold$threshold_mean), aes(MDTR, ilr2), 
    method = "gam", formula = y ~ x, se = FALSE, colour = "grey", linewidth = 2.5) +
    geom_smooth(data = ns_rf_ilr2_df, aes(MDTR, ilr2), method = "gam", formula = y ~ s(x), 
    se = FALSE, colour = "#cf9c6a", linewidth = 2, alpha = 0.5)  +
    theme_bw() +
    labs(x = "MDTR", y = "ILR2") +
    my_theme